In [14]:
import toml 
import subprocess
import os
import shutil

In [15]:
config_file = "D:/PAG/client_pag_exploratory/src/config.toml"

In [16]:
def load_toml(path: str) -> dict:
    """Load config from toml file at path."""
    with open(path, "r", encoding="utf-8") as toml_file:
        data = toml.load(toml_file)
    return data

In [17]:
config = load_toml(config_file)

In [18]:
config

{'transcad_path': 'C:/Program Files/TransCAD 6.0',
 'r_program_path': 'D:/PAG/R/R-4.3.3/bin/Rscript.exe',
 'moves4_path': 'C:/Users/Public/EPA/MOVES/MOVES4.0',
 'abm_model_path': 'D:/PAG/client_pag_abm_development/models/abm',
 'moves_input_excel_path': 'D:/PAG/client_pag_exploratory/data/external/MOVES/2035_moves4_test.xlsx',
 'out_dir': 'D:/PAG/client_pag_exploratory/data/interim',
 'src_dir': 'D:/PAG/client_pag_exploratory/src',
 'scenario_year': 2035,
 'scenario_name': 'test'}

STEP 1: Run PAG ABM

In [21]:
# Compile RSC Files
# transcad_path\rscc.exe -c -u abm_compiled_ui_dbd_path @abm_gisdk_list_file_path

rscc_path = os.path.join(config['transcad_path'], "rscc.exe").replace('/','\\')
abm_path = config['abm_model_path']
dbd_file = os.path.join(abm_path, "ui", "phoenix_ui.dbd").replace('/','\\')
lst_file = os.path.join(abm_path, "gisdk", "phoenix_ui.lst").replace('/','\\')

result = subprocess.run([rscc_path, "-c", "-u", dbd_file, "@" + lst_file])

In [22]:
# Run Macro
# transcad_path\tcw.exe -q -a abm_compiled_ui_path\phoenix_ui.dbd -ai "Run Model"

tcw_path = os.path.join(config['transcad_path'], "tcw.exe").replace('/','\\')
macro_name = "Run_Model"

result = subprocess.run([tcw_path, "-q", "-a", dbd_file, "-ai", macro_name])

STEP 2: Run Process Model Outputs GISDK

In [23]:
def update_argument_in_gisdk_file(file_path, arg, value):
    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Iterate through each line
    for i, line in enumerate(lines):
        # Check if the line starts with the specific string followed by '='
        if arg + '=' in line:
            lines[i] = '\t' + arg + '=' + "'" + value + "'" + '\n'

    # Write back to the file
    with open(file_path, 'w') as file:
        file.writelines(lines)

In [24]:
gisdk_file = os.path.join(config['src_dir'], "process_model_outputs.rsc")

update_argument_in_gisdk_file(gisdk_file, "model_folder", config['abm_model_path'].replace('/','\\\\'))
update_argument_in_gisdk_file(gisdk_file, "output_folder", config['out_dir'].replace('/','\\\\'))
update_argument_in_gisdk_file(gisdk_file, "iteration", "3")

In [25]:
# Compile RSC file

temp_dir = os.path.join(config['abm_model_path'], "ui", "temp")
if not os.path.exists(temp_dir):
    os.mkdir(temp_dir)
    
dbd_file = os.path.join(temp_dir, "temp.dbd").replace('/','\\')
rsc_file = os.path.join(config['src_dir'], "process_model_outputs.rsc").replace('/','\\')

result = subprocess.run([rscc_path, "-c", "-u", dbd_file, rsc_file])

In [26]:
# Run Macro
macro_name = "MOVES_input"
result = subprocess.run([tcw_path, "-q", "-a", dbd_file, "-ai", macro_name])

shutil.rmtree(temp_dir)

STEP 3: Create MOVES Inputs

In [27]:
results = subprocess.run(
    [
        "python",
        os.path.join(config['src_dir'], "create_moves_inputs.py")
    ],
    capture_output=True
)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: Step to create inputs using 'create_moves_inputs.py' script did not complete successfully!!!")

STEP 4: Copy MOVES Inputs to Excel

In [28]:
scenario_excel_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".xlsx")
scenario_excel_file = scenario_excel_file.replace('/','\\')
shutil.copyfile(config['moves_input_excel_path'], scenario_excel_file)

results = subprocess.run(
    [
        "python",
        os.path.join(config['src_dir'], "copy_outputs_to_excel.py"),
        scenario_excel_file,
        config['out_dir'],
        str(config['scenario_year'])
    ],
    capture_output=True
)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: Step to copy inputs to Excel spredsheet using 'copy_outputs_to_excel.py' script did not complete successfully!!!")


STEP 5: Run MOVES Model

In [29]:
# Write input database XML file 
# Rscript.exe write_moves_input_database_xml.R scenario_year, scenario_name, input_excel_file, output_dir

scenario_xml_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".xml")
#scenario_xml_file = scenario_xml_file.replace('/','\\')

results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "write_moves_input_database_xml.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        scenario_excel_file,
        scenario_xml_file
    ],
    capture_output=True
)

In [30]:
# Write run specification file
# Rscript.exe write_moves_run_spec.R scenario_year, scenario_name, output_dir

scenario_spec_file = os.path.join(config['out_dir'], str(config['scenario_year']) + "_moves4_" + config['scenario_name'] + ".mrs")
#scenario_spec_file = scenario_spec_file.replace('/','\\')

results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "write_moves_run_spec.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        scenario_spec_file
    ],
    capture_output=True
)

In [31]:
def update_path_in_batch_file(file_path, key, value):
    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Iterate through each line
    for i, line in enumerate(lines):
        # Check if the line starts with the "set key="
        if "set " + key + '=' in line:
            lines[i] = "set " + key + '=' + value + '\n'

    # Write back to the file
    with open(file_path, 'w') as file:
        file.writelines(lines)

In [33]:
# Run MOVES from a batch file

moves_batch_file = os.path.join(config['src_dir'], "run_moves.bat")

update_path_in_batch_file(moves_batch_file, "moves_dir", config['moves4_path'])
update_path_in_batch_file(moves_batch_file, "input_xml_file", scenario_xml_file)
update_path_in_batch_file(moves_batch_file, "input_spec_file", scenario_spec_file)

In [34]:
results = subprocess.run(moves_batch_file, capture_output=True, shell=True)

ret_code = results.returncode

if ret_code != 0:
    raise Exception("ERROR: MOVES run did not finish successfully!!!")

In [36]:
# Process outputs
# Rscript.exe process_moves_outputs.R scenario_year, scenario_name, output_dir, database_password
results = subprocess.run(
    [
        config['r_program_path'],
        os.path.join(config['src_dir'], "process_moves_outputs.R"),
        str(config['scenario_year']),
        config['scenario_name'],
        config['out_dir'],
        "1234"
    ],
    capture_output=True
)
